In [ ]:
import numpy as np
import pandas as pd
import re
import nltk
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv("/content/IMDB Dataset.csv")

In [ ]:
df.shape

In [ ]:
df.isnull().sum()


In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

# Pre-processing

In [ ]:
# Converting all the values into lower-case

df['review'] = df['review'].str.lower()

# removes all the links from the dataset
def remove_urls(text):
  new_text = re.sub(r"http\S+","",text)               # pattern, replace and string  s => means space and words removes
  return new_text
df['review'] = df['review'].apply(remove_urls)


# removes the punctutations marks from the dataset
def remove_punctutations(text):
  new_text =re.sub(r"[^A-Za-z0-9\s]","",text)
  return new_text
df['review'] = df['review'].apply(remove_punctutations)

# remove the html tags from the dataset
def remove_html(text):
  new_text = re.sub(r"<.*?>","",text)
  return new_text

df['review'] = df['review'].apply(remove_html)


In [ ]:
nltk.download('punkt')
nltk.download("punkt_tab")
nltk.download("stopwords")

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
# removes stop words

def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word,"")
  return text
df['review'] = df['review'].apply(remove_stopwords)

In [ ]:
df

# Stemming

In [ ]:
from nltk.stem import PorterStemmer

In [ ]:
def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []

  tokens = word_tokenize(text)

  for word in tokens:
    stemmed_token = ps.stem(word)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)

df['review'] = df['review'].apply(stemming)

In [ ]:
df.head()

# Encoding

In [ ]:
le = LabelEncoder()

df['sentiment'] = le.fit_transform(df['sentiment'])

y = df['sentiment']

In [ ]:
y

# Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df['review'])

In [ ]:
X

In [ ]:
print(X)         # Now our all the text convert to in numbers after the vectorization

# Dataset and DataLoader

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_test.shape

In [ ]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [ ]:
X_train = X_train.toarray()

In [ ]:
X_test = X_test.toarray()

In [ ]:
X_train

In [ ]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values,dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.long)

In [ ]:
train_set = TensorDataset(X_train_tensor,y_train_tensor)
test_set = TensorDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_loader = DataLoader(train_set,batch_size=128,shuffle=True)
test_loader = DataLoader(test_set,batch_size=128,shuffle=True)

# Build RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
  def __init__(self,input_size, hidden_size=128,num_layers = 1):

    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    #rnn layer
    self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

    #fc layer
    self.fc = nn.Linear(hidden_size,1)

  def forward(self, x):
    # optional => shape(num of layers, batch_size,hidden_size)
    h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

    out, _ = self.rnn(x,h0)

    out = self.fc(out[:,-1,:])

    return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training the model

In [ ]:
epochs = 10

for epoch in range(epochs):
  model.train()

  for batch_features, batch_labels in train_loader:
    optimizer.zero_grad()

    batch_features = batch_features.unsqueeze(1)         # batch features ke 3 dimension ar modde convert korte hobe tai unsqueeze apply kora hoise

    outputs = model(batch_features)

    outputs = torch.sigmoid(outputs.squeeze())    # probability

    loss = criterion(outputs, batch_labels.float()) # Convert batch_labels to float
    loss.backward()

    optimizer.step()  # Weight updates

  print(f"Epoch : {epoch+1} / {epochs} , Loss : {loss.item()}")

# Evaluate the model

In [ ]:
model.eval()

with torch.no_grad():
  correct_values = 0
  total_values = 0

  for batch_features,batch_labels in test_loader:
    batch_features = batch_features.unsqueeze(1)

    outputs = model(batch_features)
    # The predicted values should come from the model's outputs, not batch_features
    predicted = (torch.sigmoid(outputs).squeeze() > 0.5).float()

    total_values += batch_labels.size(0)
    correct_values += (predicted ==  batch_labels).sum().item()

  print(f"Accuracy : {correct_values / total_values * 100}")